# Customer Churn Prediction: Exploratory Data Analysis

This notebook performs exploratory data analysis on the Telco Customer Churn dataset to understand the patterns and factors that contribute to customer churn.

## 1. Setup and Data Loading

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

# Configure plot size and resolution
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

# Display all columns
pd.set_option('display.max_columns', None)

In [ ]:
# Add project root to path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

# Import project modules
from src.data.data_loader import download_dataset, load_raw_data
from src.config import RAW_DATA_DIR

In [ ]:
# Download the dataset if it doesn't exist
file_path = download_dataset()

# Load the dataset
df = load_raw_data(file_path)

# Display the first few rows
df.head()

## 2. Data Overview and Cleaning

In [ ]:
# Check the shape of the dataset
print(f&quot;Dataset shape: {df.shape}&quot;)

In [ ]:
# Check data types
df.dtypes

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})

# Display only columns with missing values
missing_df[missing_df['Missing Values'] > 0].sort_values('Missing Values', ascending=False)

In [ ]:
# Check the rows with missing TotalCharges
df[df['TotalCharges'].isnull()]

In [ ]:
# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing values in TotalCharges
# For customers with tenure=0, TotalCharges should be 0
df['TotalCharges'].fillna(0, inplace=True)

In [ ]:
# Check for duplicates
duplicate_count = df.duplicated().sum()
print(f&quot;Number of duplicate rows: {duplicate_count}&quot;)

In [ ]:
# Convert target variable to binary
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Check the distribution of the target variable
churn_distribution = df['Churn'].value_counts(normalize=True) * 100
print(&quot;Churn Distribution:&quot;)
print(f&quot;No Churn: {churn_distribution[0]:.2f}%&quot;)
print(f&quot;Churn: {churn_distribution[1]:.2f}%&quot;)

## 3. Exploratory Data Analysis

### 3.1 Univariate Analysis

In [ ]:
# Summary statistics for numerical variables
df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe()

In [ ]:
# Distribution of tenure
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.histplot(df['tenure'], kde=True)
plt.title('Distribution of Tenure')
plt.xlabel('Tenure (months)')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['tenure'])
plt.title('Boxplot of Tenure')
plt.ylabel('Tenure (months)')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of MonthlyCharges
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.histplot(df['MonthlyCharges'], kde=True)
plt.title('Distribution of Monthly Charges')
plt.xlabel('Monthly Charges ($)')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['MonthlyCharges'])
plt.title('Boxplot of Monthly Charges')
plt.ylabel('Monthly Charges ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of TotalCharges
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.histplot(df['TotalCharges'], kde=True)
plt.title('Distribution of Total Charges')
plt.xlabel('Total Charges ($)')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['TotalCharges'])
plt.title('Boxplot of Total Charges')
plt.ylabel('Total Charges ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of categorical variables
categorical_columns = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
                       'PhoneService', 'MultipleLines', 'InternetService',
                       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                       'TechSupport', 'StreamingTV', 'StreamingMovies',
                       'Contract', 'PaperlessBilling', 'PaymentMethod']

# Create a figure with subplots
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

# Plot each categorical variable
for i, column in enumerate(categorical_columns):
    if i < len(axes):
        sns.countplot(x=column, data=df, ax=axes[i])
        axes[i].set_title(f'Distribution of {column}')
        axes[i].set_xlabel('')
        axes[i].tick_params(axis='x', rotation=45)

# Remove empty subplots
for i in range(len(categorical_columns), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

### 3.2 Bivariate Analysis with Target Variable

In [ ]:
# Numerical variables vs Churn
plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
sns.boxplot(x='Churn', y='tenure', data=df)
plt.title('Tenure vs Churn')
plt.xlabel('Churn (1=Yes, 0=No)')
plt.ylabel('Tenure (months)')

plt.subplot(1, 3, 2)
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title('Monthly Charges vs Churn')
plt.xlabel('Churn (1=Yes, 0=No)')
plt.ylabel('Monthly Charges ($)')

plt.subplot(1, 3, 3)
sns.boxplot(x='Churn', y='TotalCharges', data=df)
plt.title('Total Charges vs Churn')
plt.xlabel('Churn (1=Yes, 0=No)')
plt.ylabel('Total Charges ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Function to plot categorical variables vs Churn
def plot_categorical_vs_churn(columns, df, rows=2):
    cols = len(columns) // rows + (1 if len(columns) % rows != 0 else 0)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*4))
    axes = axes.flatten()
    
    for i, column in enumerate(columns):
        if i < len(axes):
            # Calculate churn rate for each category
            churn_rate = df.groupby(column)['Churn'].mean().sort_values(ascending=False)
            
            # Plot churn rate
            churn_rate.plot(kind='bar', ax=axes[i])
            axes[i].set_title(f'Churn Rate by {column}')
            axes[i].set_xlabel('')
            axes[i].set_ylabel('Churn Rate')
            axes[i].tick_params(axis='x', rotation=45)
            
            # Add value labels on top of bars
            for j, v in enumerate(churn_rate):
                axes[i].text(j, v + 0.02, f'{v:.2f}', ha='center')
    
    # Remove empty subplots
    for i in range(len(columns), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot demographic variables vs Churn
demographic_columns = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']
plot_categorical_vs_churn(demographic_columns, df, rows=1)

In [ ]:
# Plot service variables vs Churn
service_columns = ['PhoneService', 'MultipleLines', 'InternetService',
                   'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                   'TechSupport', 'StreamingTV', 'StreamingMovies']
plot_categorical_vs_churn(service_columns, df, rows=3)

In [ ]:
# Plot account variables vs Churn
account_columns = ['Contract', 'PaperlessBilling', 'PaymentMethod']
plot_categorical_vs_churn(account_columns, df, rows=1)

### 3.3 Correlation Analysis

In [ ]:
# Convert categorical variables to dummy variables for correlation analysis
df_encoded = pd.get_dummies(df.drop('customerID', axis=1), drop_first=True)

# Calculate correlation matrix
correlation_matrix = df_encoded.corr()

# Plot correlation heatmap
plt.figure(figsize=(16, 14))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Get correlations with Churn
churn_correlations = correlation_matrix['Churn'].sort_values(ascending=False)

# Plot top positive and negative correlations
plt.figure(figsize=(12, 10))
churn_correlations.drop('Churn').abs().sort_values(ascending=False).head(15).plot(kind='bar')
plt.title('Top 15 Features Correlated with Churn')
plt.xlabel('Features')
plt.ylabel('Absolute Correlation')
plt.tight_layout()
plt.show()

In [ ]:
# Display top positive correlations with Churn
print(&quot;Top Positive Correlations with Churn:&quot;)
print(churn_correlations.drop('Churn').head(10))

In [ ]:
# Display top negative correlations with Churn
print(&quot;Top Negative Correlations with Churn:&quot;)
print(churn_correlations.drop('Churn').tail(10).iloc[::-1])

### 3.4 Feature Engineering Ideas

In [ ]:
# Create average monthly charge feature
df['avg_monthly_charges'] = df['TotalCharges'] / df['tenure'].replace(0, 1)

# Create binary feature for long-term customers
df['is_long_term'] = (df['tenure'] > 24).astype(int)

# Create binary feature for high-value customers
df['is_high_value'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)

# Count the number of services each customer has
service_columns = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                   'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

# Count services that are not 'No' or 'No internet service'
df['service_count'] = 0
for col in service_columns:
    df['service_count'] += ((df[col] != 'No') & (df[col] != 'No internet service')).astype(int)

In [ ]:
# Analyze new features vs Churn
plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
sns.boxplot(x='Churn', y='avg_monthly_charges', data=df)
plt.title('Average Monthly Charges vs Churn')
plt.xlabel('Churn (1=Yes, 0=No)')
plt.ylabel('Average Monthly Charges ($)')

plt.subplot(1, 3, 2)
sns.countplot(x='is_long_term', hue='Churn', data=df)
plt.title('Long-term Customer vs Churn')
plt.xlabel('Is Long-term Customer (>24 months)')
plt.ylabel('Count')
plt.legend(title='Churn', labels=['No', 'Yes'])

plt.subplot(1, 3, 3)
sns.boxplot(x='Churn', y='service_count', data=df)
plt.title('Service Count vs Churn')
plt.xlabel('Churn (1=Yes, 0=No)')
plt.ylabel('Number of Services')

plt.tight_layout()
plt.show()

## 4. Key Insights and Conclusions

### 4.1 Summary of Findings

1. **Churn Rate**: The overall churn rate in the dataset is approximately 26.5%, indicating that about a quarter of customers leave the service.

2. **Tenure Impact**: There is a strong negative correlation between tenure and churn. Customers with longer tenure are less likely to churn.

3. **Contract Type**: Contract type has a significant impact on churn. Month-to-month contracts have a much higher churn rate compared to one-year or two-year contracts.

4. **Payment Method**: Electronic check payment method is associated with higher churn rates compared to automatic payment methods.

5. **Services**: Customers without online security, tech support, and device protection are more likely to churn.

6. **Internet Service**: Fiber optic internet service has a higher churn rate compared to DSL.

7. **Monthly Charges**: Higher monthly charges are associated with higher churn rates.

8. **Service Count**: Customers with fewer services are more likely to churn.

9. **Senior Citizens**: Senior citizens have a slightly higher churn rate.

10. **Paperless Billing**: Customers with paperless billing are more likely to churn.

### 4.2 Recommendations for Feature Engineering

Based on the exploratory data analysis, the following feature engineering steps are recommended:

1. **Interaction Features**:
   - Create interaction between tenure and monthly charges
   - Create interaction between contract type and monthly charges

2. **Ratio Features**:
   - Average monthly charge (total charges / tenure)
   - Service value ratio (monthly charges / service count)

3. **Binary Features**:
   - Long-term customer indicator (tenure > 24 months)
   - High-value customer indicator (monthly charges > median)
   - Multiple services indicator (service count > 3)

4. **Aggregated Features**:
   - Total number of services
   - Total number of security services (online security, tech support, device protection)
   - Total number of entertainment services (streaming TV, streaming movies)

5. **Categorical Encoding**:
   - One-hot encoding for categorical variables
   - Target encoding for high-cardinality categorical variables

### 4.3 Next Steps

1. **Data Preprocessing**:
   - Handle missing values
   - Encode categorical variables
   - Scale numerical features

2. **Feature Engineering**:
   - Implement the recommended feature engineering steps
   - Evaluate the impact of new features on model performance

3. **Model Development**:
   - Train baseline models
   - Perform hyperparameter tuning
   - Evaluate model performance

4. **Model Interpretation**:
   - Analyze feature importance
   - Understand the factors driving churn
   - Provide actionable insights